In [1]:
import pandas as pd
import numpy as np

In [2]:
import openpyxl

In [3]:
df = pd.read_excel("E:/Mealawe-Kitchen-Vendor-Automation-Project/Files/FoodOrdersData.xlsx")   

# Columns you want to extract
required_columns = [
    "itemList[0].itemDescription",
    "addOns[0].addOnName",
    "addOns[1].addOnName",
    "addOns[2].addOnName",
    "addOns[3].addOnName",
    "addOns[0].count",
    "addOns[1].count",
    "addOns[2].count",
    "addOns[3].count",
    "beforePackingImageUrl"
]

In [4]:
df.head()

,_id,kitchenName,itemList[0].itemName,itemList[0].itemType,itemList[0].itemDescription,addOns[0].addOnName,addOns[1].addOnName,addOns[2].addOnName,addOns[3].addOnName,addOns[0].count,addOns[1].count,addOns[2].count,addOns[3].count,packageName,orderNo,beforePackingImageUrl,afterPackingImageUrl
0,69453d9c7da649068a39c2a7,OYO BLR3220,Deluxe - Weekly,NonVeg,"3 Chapati, 1 Chicken Curry, Rice, Cut Salad",2 Chapati Extra,NaN,NaN,NaN,1.0,NaN,NaN,NaN,Deluxe - Weekly,70000000600,https://api.mealawe.com/images/fffd92182a744a9...,https://api.mealawe.com/images/4c4aff6e86c1401...
1,69021bfa20ec31396463409c,Pune Cloud Kitchen,Comfort Thali - Monthly,Veg,"1 Veg Curry, Rice, Dal, Cut Salad",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comfort Thali - Monthly,20015850,https://api.mealawe.com/images/fffbea48151f43b...,https://api.mealawe.com/images/60e56f617f4b4eb...
2,68d74c54cd4de2392d9c298e,RZ Homely Kitchen,Standard - Monthly,Veg,"3 Chapati, 1 Veg Curry, Cut Salad",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard - Monthly,20005115,https://api.mealawe.com/images/fffa26f8f8ed456...,https://api.mealawe.com/images/73982abfbb474fa...
3,68de7401a4b6c8392f3c3343,Anku's Kitchen,Standard - Biweekly,Veg,"3 Chapati, 1 Veg Curry, Cut Salad",1 Chapati Extra,NaN,NaN,NaN,1.0,NaN,NaN,NaN,Standard - Biweekly,20007409,https://api.mealawe.com/images/fff6645dfea94d0...,https://api.mealawe.com/images/b5a6430c2c55496...
4,6929dbcff1fe574077513ca6,Green wish kitchen,Jain Deluxe - Monthly,Veg,"3 Chapati/Roti, 1 Jain Curry/Dry, 1 Dal, 1 Ste...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Jain Deluxe - Monthly,20030917,https://api.mealawe.com/images/fff4b6c5ce0a47e...,https://api.mealawe.com/images/64abdf4362834df...


In [5]:
new_df = df[required_columns]

with pd.ExcelWriter("E:/Mealawe-Kitchen-Vendor-Automation-Project/Files/output.xlsx", engine="openpyxl") as writer:
    new_df.to_excel(writer, sheet_name="Extracted_Data", index=False)

print("✅ New sheet created successfully!")

✅ New sheet created successfully!


In [6]:
sample_df = new_df.head()

with pd.ExcelWriter("E:/Mealawe-Kitchen-Vendor-Automation-Project/Files/sample_data.xlsx", engine="openpyxl") as writer:
    sample_df.to_excel(writer, sheet_name="Extracted_Data", index=False)


In [8]:
df = pd.read_excel("E:/Mealawe-Kitchen-Vendor-Automation-Project/Files/output.xlsx")

In [9]:
# Get unique image URLs
unique_urls = df["beforePackingImageUrl"].dropna().unique()

# Assign IDs
image_id_map = {
    url: idx + 1
    for idx, url in enumerate(unique_urls)
}


In [10]:
df["image_id"] = df["beforePackingImageUrl"].map(image_id_map)


In [11]:
# Re-structuring addon cols
def extract_addons(row):
    addons = []
    for i in range(4):
        name = row.get(f"addOns[{i}].addOnName")
        count = row.get(f"addOns[{i}].count")

        if pd.notna(name):
            addons.append(f"{name}:{int(count)}")
    return ", ".join(addons)


In [12]:
final_df = pd.DataFrame({
    "image_id": df["image_id"],
    "item_description": df["itemList[0].itemDescription"],
    "addons": df.apply(extract_addons, axis=1)
})


In [13]:
final_df.head()

,image_id,item_description,addons
0,1,"3 Chapati, 1 Chicken Curry, Rice, Cut Salad",2 Chapati Extra:1
1,2,"1 Veg Curry, Rice, Dal, Cut Salad",
2,3,"3 Chapati, 1 Veg Curry, Cut Salad",
3,4,"3 Chapati, 1 Veg Curry, Cut Salad",1 Chapati Extra:1
4,5,"3 Chapati/Roti, 1 Jain Curry/Dry, 1 Dal, 1 Ste...",


In [14]:
image_map_df = pd.DataFrame({
    "image_id": image_id_map.values(),
    "image_url": image_id_map.keys()
})


In [15]:
with pd.ExcelWriter("E:/Mealawe-Kitchen-Vendor-Automation-Project/Files/processed_dataset.xlsx", engine="openpyxl") as writer:
    final_df.to_excel(writer, sheet_name="dataset", index=False)
    image_map_df.to_excel(writer, sheet_name="image_id_mapping", index=False)

print("✅ Dataset created successfully!")


✅ Dataset created successfully!


In [16]:
import os
import requests

In [23]:
IMAGE_DIR = "E:/Mealawe-Kitchen-Vendor-Automation-Project/Files/meal_images"
os.makedirs(IMAGE_DIR, exist_ok=True)


In [21]:
def download_image(url, image_id):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image_path = os.path.join(IMAGE_DIR, f"{image_id}.jpg")

        with open(image_path, "wb") as f:
            f.write(response.content)

        return True
    except Exception:
        print(f"❌ Failed to download {url} | ID {image_id}")
        return False


In [20]:
import random

MAX_IMAGES = 1000

all_image_ids = list(image_id_map.values())

selected_image_ids = set(
    random.sample(all_image_ids, min(MAX_IMAGES, len(all_image_ids)))
)


In [ ]:
downloaded_image_ids = set()

for url, image_id in image_id_map.items():
    if image_id in selected_image_ids:
        success = download_image(url, image_id)
        if success:
            downloaded_image_ids.add(image_id)


In [ ]:
subset_final_df = final_df[final_df["image_id"].isin(downloaded_image_ids)].copy()
subset_image_map_df = image_map_df[image_map_df["image_id"].isin(downloaded_image_ids)].copy()


In [ ]:
with pd.ExcelWriter("E:/Mealawe-Kitchen-Vendor-Automation-Project/Files/processed_dataset_subset.xlsx", engine="openpyxl") as writer:
    subset_final_df.to_excel(writer, sheet_name="dataset", index=False)
    subset_image_map_df.to_excel(writer, sheet_name="image_id_mapping", index=False)

print("✅ Dataset created successfully!")
